<a href="https://colab.research.google.com/github/poorvigupta26/Natural-Language-to-Structured-Function-Pipeline-using-Open-Source-LLM/blob/main/nlp_to_fn_pipeline_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#rough outline

## 1. Install & Import Dependencies
## 2. Define the Function Library (JSON format)
## 3. Define Prompt Template for LLM
## 4. Connect to LLM via API (Together.ai / OpenRouter)
## 5. Query Planning Logic
## 6. Example Inputs & Outputs
## 7. JSON Output Formatter

In [6]:
import json
import os
import re
from datetime import datetime

In [7]:
function_library = [
    {
        "name": "get_invoices",
        "description": "Fetches all invoice records for a given month.",
        "inputs": {
            "month": "string"
        },
        "outputs": {
            "invoices": "list"
        }
    },
    {
        "name": "summarize_invoices",
        "description": "Summarizes a list of invoices to get total amount and stats.",
        "inputs": {
            "invoices": "list"
        },
        "outputs": {
            "summary": "string"
        }
    },
    {
        "name": "send_email",
        "description": "Sends an email with subject and body to a recipient.",
        "inputs": {
            "to": "string",
            "subject": "string",
            "body": "string"
        },
        "outputs": {
            "status": "string"
        }
    }
]

with open("function_library.json", "w") as f:
    json.dump(function_library, f, indent=2)

print("function_library created")


function_library created


In [8]:
def format_function_prompt(function_registry):
    prompt = ""
    for fn in function_registry:
        name = fn["name"]
        desc = fn["description"]
        inputs = fn["inputs"]
        inputs_str = ", ".join(f"{k}: {v}" for k, v in inputs.items())
        prompt += f"- {name}({inputs_str}): {desc}\n"
    return prompt


print(format_function_prompt(function_library));

- get_invoices(month: string): Fetches all invoice records for a given month.
- summarize_invoices(invoices: list): Summarizes a list of invoices to get total amount and stats.
- send_email(to: string, subject: string, body: string): Sends an email with subject and body to a recipient.



In [9]:
def build_llm_prompt(user_query, function_registry):
    function_list_str = format_function_prompt(function_registry)

    prompt = f"""
You are a function planner AI. Your task is to read the user's query and break it down into a list of structured function calls.

Use ONLY the functions from the list below. Each function has a name and input arguments:

{function_list_str}

---

Analyze the user's query below and generate a JSON array representing a plan — a step-by-step ordered list of function calls. Each function call must include:
- the function name
- the dictionary of arguments (key: value)

Use placeholder strings like "<output_of_function_name>" when chaining outputs.

Respond ONLY with a valid JSON array of function calls like:
[
  {{
    "function": "function_name",
    "inputs": {{"arg1": "value1"}}
  }},
  ...
]

DO NOT ECHO BACK THE PROMPT ITSELF

User:
\"{user_query}\"

Plan:
"""
    return prompt


In [10]:
model = "mistralai/Mistral-7B-Instruct-v0.1"

In [11]:
prompt = build_llm_prompt("Get March invoices and send summary to ayush@example.com", function_library)


In [12]:
from google.colab import userdata
hf_key = userdata.get('HF_TOKEN')

In [13]:
!pip install huggingface_hub
from huggingface_hub import InferenceClient

client = InferenceClient(model="mistralai/Mistral-7B-v0.1", token=hf_key)

In [2]:
!pip install transformers accelerate bitsandbytes

In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "mistralai/Mistral-7B-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_4bit=True
)


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [16]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/usr/local/lib/python3.11/dist-packages/bitsandbytes/nn/modules.py:463: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(



You are a function planner AI. Your task is to read the user's query and break it down into a list of structured function calls.

Use ONLY the functions from the list below. Each function has a name and input arguments:

- get_invoices(month: string): Fetches all invoice records for a given month.
- summarize_invoices(invoices: list): Summarizes a list of invoices to get total amount and stats.
- send_email(to: string, subject: string, body: string): Sends an email with subject and body to a recipient.


--- 

Analyze the user's query below and generate a JSON array representing a plan — a step-by-step ordered list of function calls. Each function call must include:
- the function name
- the dictionary of arguments (key: value)

Use placeholder strings like "<output_of_function_name>" when chaining outputs.

Respond ONLY with a valid JSON array of function calls like:
[
  {
    "function": "function_name",
    "inputs": {"arg1": "value1"}
  },
  ...
]

DO NOT ECHO BACK THE PROMPT ITSE